In [ ]:
# 셀 1: Foursquare NYC 체크인 데이터 확보
# 방법 A) 직접 URL이 있으면 그걸로. 없으면 방법 B(수동 업로드) 사용.
import os, urllib.request

DATA_FILE = "dataset_TSMC2014_NYC.txt"

if not os.path.exists(DATA_FILE):
    print("파일이 없습니다. 아래 중 하나로 받아주세요:")
    print(" - Kaggle: 'foursquare-nyc-and-tokyo-checkin-dataset' 에서")
    print("   dataset_TSMC2014_NYC.txt 다운로드 후, Colab 좌측 파일창에 업로드")
    print(" - 또는 원저자(Dingqi Yang) 사이트의 TSMC2014 zip 내 NYC 파일")
    # Colab에서 수동 업로드를 쓰려면 아래 두 줄 주석 해제:
    # from google.colab import files
    # files.upload()
else:
    print("이미 있음:", DATA_FILE)

In [ ]:
# 셀 1-A: 데이터 다운로드 시도 (공개 미러)
import os, urllib.request, zipfile

DATA_FILE = "dataset_TSMC2014_NYC.txt"

if not os.path.exists(DATA_FILE):
    url = "http://www-public.tem-tsp.eu/~zhang_da/pub/dataset_tsmc2014.zip"
    try:
        print("다운로드 시도 중…")
        urllib.request.urlretrieve(url, "tsmc2014.zip")
        with zipfile.ZipFile("tsmc2014.zip") as z:
            z.extractall(".")
            print("압축 해제됨:", z.namelist())
    except Exception as e:
        print("자동 다운로드 실패:", e)
        print("→ 아래 '수동 업로드' 방법을 쓰세요.")

# 파일 위치 확인 (하위 폴더에 풀렸을 수 있음)
import glob
hits = glob.glob("**/dataset_TSMC2014_NYC.txt", recursive=True)
print("찾은 파일:", hits)

In [ ]:
# 셀 1-B: 수동 업로드
from google.colab import files
files.upload()   # 파일 선택창 → dataset_TSMC2014_NYC.txt 선택

In [ ]:
# 진단: NYC 파일이 어디에 있는지 확인
import glob, os
print("=== dataset_TSMC2014_NYC.txt 검색 ===")
print(glob.glob("**/dataset_TSMC2014_NYC.txt", recursive=True))
print("\n=== 현재 폴더의 txt/zip 파일들 ===")
print([f for f in os.listdir(".") if f.endswith((".txt", ".zip"))])
print("\n=== TSMC 관련 모든 파일 ===")
print(glob.glob("**/*TSMC*", recursive=True))

In [ ]:
# 파일 실제 형식 확인 (헤더·구분자·컬럼)
DATA_FILE = "dataset_TSMC2014_NYC.csv"

with open(DATA_FILE, encoding="latin-1") as f:
    for i in range(3):
        print(repr(f.readline()))

In [ ]:
# 셀 2 (csv 버전): 헤더 있고 쉼표 구분이라고 가정
import pandas as pd

DATA_FILE = "dataset_TSMC2014_NYC.csv"
df = pd.read_csv(DATA_FILE, encoding="latin-1")

print("컬럼:", list(df.columns))   # ← 실제 컬럼명 확인용
print("체크인 수:", len(df))
df.head()

In [ ]:
# 셀 3+4: 시간 파싱 → userId별 시퀀스 → train/test 분할
import pandas as pd

# (df 는 직전 셀에서 이미 로드됨)
df["time"] = pd.to_datetime(df["utcTimestamp"], format="%a %b %d %H:%M:%S %z %Y")

# 사용자별 시간순 정렬 → 방문 시퀀스 (우리 앱의 A→B→C)
df = df.sort_values(["userId", "time"])
sequences = df.groupby("userId")["venueId"].apply(list)

seq_len = sequences.apply(len)
print("사용자당 방문 수 — 중앙값:", int(seq_len.median()), "/ 최대:", int(seq_len.max()))

# 너무 짧은 시퀀스 제외 (최소 5회)
MIN_LEN = 5
sequences = sequences[seq_len >= MIN_LEN]
print(f"학습 대상 사용자: {len(sequences)}명 (방문 {MIN_LEN}회 이상)")

# train: 각 시점의 (앞부분 → 다음 한 곳) / test: 마지막 1개 (leave-one-out)
train_samples, test_samples = [], []
for uid, seq in sequences.items():
    for i in range(1, len(seq) - 1):
        train_samples.append({"user": uid, "history": seq[:i], "target": seq[i]})
    test_samples.append({"user": uid, "history": seq[:-1], "target": seq[-1]})

print("train 샘플:", len(train_samples))
print("test 샘플:", len(test_samples), "(사용자당 1개)")
print("\n예시 train 샘플:", {**train_samples[0], "history": train_samples[0]["history"][:3]}, "...")

In [ ]:
# 노트북 2 - 셀 1: 평가지표 정의 + 인기순 베이스라인
import numpy as np
from collections import Counter

# --- 평가지표 ---
def recall_at_k(ranked, target, k):
    return 1.0 if target in ranked[:k] else 0.0

def ndcg_at_k(ranked, target, k):
    for i, v in enumerate(ranked[:k]):
        if v == target:
            return 1.0 / np.log2(i + 2)  # i=0 → 1.0
    return 0.0

def evaluate(rank_fn, test_samples, ks=(1, 5, 10)):
    """rank_fn(sample) -> 추천 venueId 리스트(상위순). 지표 평균 반환."""
    metrics = {f"Recall@{k}": [] for k in ks}
    metrics.update({f"NDCG@{k}": [] for k in ks})
    for s in test_samples:
        ranked = rank_fn(s)
        for k in ks:
            metrics[f"Recall@{k}"].append(recall_at_k(ranked, s["target"], k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(ranked, s["target"], k))
    return {m: round(float(np.mean(v)), 4) for m, v in metrics.items()}

# --- 베이스라인 1: 전역 인기순 (train 전체에서 가장 많이 방문된 POI 순) ---
pop_counter = Counter()
for s in train_samples:
    pop_counter[s["target"]] += 1
popular_ranked = [v for v, _ in pop_counter.most_common()]

def rank_popular(sample):
    # 사용자가 이미 방문한 곳은 제외하고 인기순
    seen = set(sample["history"])
    return [v for v in popular_ranked if v not in seen]

res_pop = evaluate(rank_popular, test_samples)
print("=== 베이스라인: 인기순(most-popular) ===")
for m, v in res_pop.items():
    print(f"  {m}: {v}")

In [ ]:
# 노트북 2 - 셀 2: 베이스라인 2 (개인 방문 이력 순) + 베이스라인 3 (개인+전역 결합)
from collections import Counter

# 베이스라인 2: 개인 이력 순 (history에서 많이 간 곳 순)
def rank_personal(sample):
    c = Counter(sample["history"])
    return [v for v, _ in c.most_common()]

res_personal = evaluate(rank_personal, test_samples)
print("=== 베이스라인 2: 개인 이력 순 ===")
for m, v in res_personal.items():
    print(f"  {m}: {v}")

# 베이스라인 3: 개인 이력 순 → 부족하면 전역 인기순으로 채움 (현실적 결합)
def rank_personal_then_popular(sample):
    seen = set(sample["history"])
    personal = [v for v, _ in Counter(sample["history"]).most_common()]
    backfill = [v for v in popular_ranked if v not in seen]
    return personal + backfill

res_combo = evaluate(rank_personal_then_popular, test_samples)
print("\n=== 베이스라인 3: 개인 이력 + 전역 인기 backfill ===")
for m, v in res_combo.items():
    print(f"  {m}: {v}")

In [ ]:
# 노트북 3 (NN L1) - 셀 1: 피처 인코딩 + 텐서 준비
import numpy as np
import torch
from collections import Counter

# --- 후보 POI를 상위 N개로 제한 (경량화) ---
TOP_N = 5000
top_pois = [v for v, _ in pop_counter.most_common(TOP_N)]
poi2idx = {v: i for i, v in enumerate(top_pois)}   # POI → 0..N-1
n_poi = len(poi2idx)
print(f"후보 POI: {n_poi}개 (상위 {TOP_N})")

# --- 각 POI의 카테고리 매핑 (입력 피처용) ---
venue_cat = df.drop_duplicates("venueId").set_index("venueId")["venueCategory"]
cats = sorted(df["venueCategory"].unique())
cat2idx = {c: i for i, c in enumerate(cats)}
n_cat = len(cat2idx)
print(f"카테고리: {n_cat}종")

def poi_cat_idx(venue):
    c = venue_cat.get(venue, None)
    return cat2idx.get(c, 0)

# --- 학습 샘플을 텐서로: (마지막방문 POI, 그 카테고리, 시간대) → 정답 POI ---
# 정답이 top_pois 안에 있는 샘플만 사용 (제한된 후보 집합 학습)
def build_xy(samples, with_time_lookup):
    last_poi, last_cat, hour, target = [], [], [], []
    for s in samples:
        tgt = s["target"]
        if tgt not in poi2idx:
            continue
        lp = s["history"][-1]
        if lp not in poi2idx:
            continue
        last_poi.append(poi2idx[lp])
        last_cat.append(poi_cat_idx(lp))
        hour.append(with_time_lookup.get((s["user"], tgt), 12))  # 없으면 정오
        target.append(poi2idx[tgt])
    return (torch.tensor(last_poi), torch.tensor(last_cat),
            torch.tensor(hour), torch.tensor(target))

# 시간대 룩업: (user, venue)별 평균 방문 시각(hour)
df["hour"] = df["time"].dt.hour
time_lookup = df.groupby(["userId", "venueId"])["hour"].first().to_dict()

Xtr_poi, Xtr_cat, Xtr_hour, ytr = build_xy(train_samples, time_lookup)
Xte_poi, Xte_cat, Xte_hour, yte = build_xy(test_samples, time_lookup)
print(f"학습 텐서: {len(ytr)}개 / 평가 텐서: {len(yte)}개")
print("예시:", Xtr_poi[0].item(), Xtr_cat[0].item(), Xtr_hour[0].item(), "→", ytr[0].item())

In [ ]:
# 노트북 3 (NN L1) - 셀 2: MLP 랭커 정의 + 학습 루프
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

class MLPRanker(nn.Module):
    def __init__(self, n_poi, n_cat, emb=64, hidden=128):
        super().__init__()
        self.poi_emb = nn.Embedding(n_poi, emb)
        self.cat_emb = nn.Embedding(n_cat, 16)
        self.hour_emb = nn.Embedding(24, 8)
        self.mlp = nn.Sequential(
            nn.Linear(emb + 16 + 8, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_poi),   # POI별 점수
        )
    def forward(self, poi, cat, hour):
        x = torch.cat([self.poi_emb(poi), self.cat_emb(cat), self.hour_emb(hour)], dim=1)
        return self.mlp(x)

model = MLPRanker(n_poi, n_cat).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()   # 정답 POI를 높게

ds = TensorDataset(Xtr_poi, Xtr_cat, Xtr_hour, ytr)
dl = DataLoader(ds, batch_size=512, shuffle=True)

EPOCHS = 15
for ep in range(EPOCHS):
    model.train(); total = 0
    for bp, bc, bh, by in dl:
        bp, bc, bh, by = bp.to(device), bc.to(device), bh.to(device), by.to(device)
        opt.zero_grad()
        out = model(bp, bc, bh)
        loss = loss_fn(out, by)
        loss.backward(); opt.step()
        total += loss.item() * len(by)
    print(f"epoch {ep+1:2d}  loss {total/len(ytr):.4f}")

In [ ]:
# 노트북 3 (NN L1) - 셀 3: 공정 평가 (동일 403 집합에서 베이스라인 vs NN)
import numpy as np

# NN 평가에 쓰인 것과 동일한 test 샘플만 추림 (정답이 top_pois 안 + 마지막방문도 안)
eval_samples = [s for s in test_samples
                if s["target"] in poi2idx and s["history"][-1] in poi2idx]
print("공정 비교 대상:", len(eval_samples), "개")

# --- 베이스라인들을 같은 집합에서 재측정 ---
print("\n[동일 403 집합 기준]")
print("개인 이력 순:", evaluate(rank_personal, eval_samples))
print("인기순     :", evaluate(rank_popular, eval_samples))

# --- NN 평가 ---
idx2poi = {i: v for v, i in poi2idx.items()}
model.eval()
def rank_nn(sample):
    lp = poi2idx[sample["history"][-1]]
    lc = poi_cat_idx(sample["history"][-1])
    h = time_lookup.get((sample["user"], sample["target"]), 12)
    with torch.no_grad():
        scores = model(torch.tensor([lp]), torch.tensor([lc]), torch.tensor([h]))[0]
    order = torch.argsort(scores, descending=True).tolist()
    return [idx2poi[i] for i in order]

print("NN L1(MLP) :", evaluate(rank_nn, eval_samples))

In [ ]:
# 노트북 4 (NN L2) - 셀 1: 사용자 이력 집계 입력 추가 + 모델 학습
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter

# --- 각 학습/평가 샘플에 "사용자 이력 POI 인덱스 집합" 추가 ---
# history 중 top_pois 안에 든 것들의 idx 리스트 (취향 요약용)
def hist_idxs(history, max_len=50):
    idxs = [poi2idx[v] for v in history if v in poi2idx]
    return idxs[-max_len:] if idxs else [0]

def build_xy2(samples):
    last_poi, last_cat, hour, target, hist = [], [], [], [], []
    for s in samples:
        tgt = s["target"]
        if tgt not in poi2idx or s["history"][-1] not in poi2idx:
            continue
        last_poi.append(poi2idx[s["history"][-1]])
        last_cat.append(poi_cat_idx(s["history"][-1]))
        hour.append(time_lookup.get((s["user"], tgt), 12))
        target.append(poi2idx[tgt])
        hist.append(hist_idxs(s["history"]))
    return last_poi, last_cat, hour, target, hist

tr = build_xy2(train_samples)
print("L2 학습 샘플:", len(tr[3]))

# 가변 길이 history → 패딩(0)으로 텐서화
def pad(hists, L=50):
    out = torch.zeros(len(hists), L, dtype=torch.long)
    for i, h in enumerate(hists):
        h = h[:L]
        out[i, :len(h)] = torch.tensor(h)
    return out

Xtr_poi2 = torch.tensor(tr[0]); Xtr_cat2 = torch.tensor(tr[1])
Xtr_hour2 = torch.tensor(tr[2]); ytr2 = torch.tensor(tr[3])
Xtr_hist2 = pad(tr[4])

class MLPRankerV2(nn.Module):
    def __init__(self, n_poi, n_cat, emb=64, hidden=128):
        super().__init__()
        self.poi_emb = nn.Embedding(n_poi, emb, padding_idx=0)
        self.cat_emb = nn.Embedding(n_cat, 16)
        self.hour_emb = nn.Embedding(24, 8)
        # 입력: 마지막POI emb + 카테고리 + 시간 + "이력 평균 emb"
        self.mlp = nn.Sequential(
            nn.Linear(emb + 16 + 8 + emb, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_poi),
        )
    def forward(self, poi, cat, hour, hist):
        h_emb = self.poi_emb(hist)                 # (B, L, emb)
        mask = (hist > 0).unsqueeze(-1).float()
        hist_avg = (h_emb * mask).sum(1) / mask.sum(1).clamp(min=1)  # 평균(패딩 제외)
        x = torch.cat([self.poi_emb(poi), self.cat_emb(cat), self.hour_emb(hour), hist_avg], dim=1)
        return self.mlp(x)

model2 = MLPRankerV2(n_poi, n_cat)
opt = torch.optim.Adam(model2.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

ds = TensorDataset(Xtr_poi2, Xtr_cat2, Xtr_hour2, Xtr_hist2, ytr2)
dl = DataLoader(ds, batch_size=512, shuffle=True)

for ep in range(15):
    model2.train(); total = 0
    for bp, bc, bh, bhist, by in dl:
        opt.zero_grad()
        out = model2(bp, bc, bh, bhist)
        loss = loss_fn(out, by)
        loss.backward(); opt.step()
        total += loss.item() * len(by)
    print(f"epoch {ep+1:2d}  loss {total/len(ytr2):.4f}")

In [ ]:
# 노트북 4 (NN L2) - 셀 2: 평가 (동일 403 집합)
model2.eval()
def rank_nn2(sample):
    lp = poi2idx[sample["history"][-1]]
    lc = poi_cat_idx(sample["history"][-1])
    h = time_lookup.get((sample["user"], sample["target"]), 12)
    hist = pad([hist_idxs(sample["history"])])
    with torch.no_grad():
        scores = model2(torch.tensor([lp]), torch.tensor([lc]), torch.tensor([h]), hist)[0]
    order = torch.argsort(scores, descending=True).tolist()
    return [idx2poi[i] for i in order]

print("=== 동일 403 집합 최종 대결 ===")
print("인기순     :", evaluate(rank_popular, eval_samples))
print("개인 이력  :", evaluate(rank_personal, eval_samples))
print("NN L1(MLP) :", evaluate(rank_nn, eval_samples))
print("NN L2(+이력):", evaluate(rank_nn2, eval_samples))

In [ ]:
# 노트북 5 (고전 ML) - 셀 1: (맥락,후보) 쌍 피처 + negative sampling
import numpy as np
from collections import Counter

# 후보 POI별 전역 인기도(=train에서 target으로 등장 횟수) — 피처로 사용
pop_count = Counter(s["target"] for s in train_samples)
max_pop = max(pop_count.values())

# 사용자별 방문 카운트(개인 이력 신호) 빠른 조회용
def user_visit_counter(history):
    return Counter(history)

def pair_features(sample, cand):
    """(맥락, 후보 POI) → 숫자 피처 벡터"""
    last = sample["history"][-1]
    uc = sample["_uc"]  # 미리 계산한 user_visit_counter
    return [
        pop_count.get(cand, 0) / max_pop,        # 후보 전역 인기도
        uc.get(cand, 0),                          # 사용자가 그 후보를 과거 방문한 횟수(단골 신호)
        1.0 if cand in uc else 0.0,               # 과거 방문 여부(0/1)
        poi_cat_idx(cand),                        # 후보 카테고리 idx
        poi_cat_idx(last),                        # 마지막 방문 카테고리 idx
        1.0 if poi_cat_idx(cand) == poi_cat_idx(last) else 0.0,  # 같은 카테고리?
        len(sample["history"]),                   # 사용자 이력 길이(활동성)
        time_lookup.get((sample["user"], cand), 12) / 24.0,      # 후보 방문 시간대(정규화)
    ]
N_FEAT = 8
NEG = 4  # 양성 1개당 음성 4개

# 학습셋 구성: 정답=1, 무작위 후보(top_pois에서)=0
def build_classic_xy(samples, neg=NEG, cap=60000):
    X, y = [], []
    pool = top_pois  # 음성 샘플링 풀(상위 5000)
    for s in samples[:cap]:  # 속도 위해 일부만(필요시 cap 조정)
        if s["target"] not in poi2idx or s["history"][-1] not in poi2idx:
            continue
        s["_uc"] = user_visit_counter(s["history"])
        # 양성
        X.append(pair_features(s, s["target"])); y.append(1)
        # 음성 neg개
        for _ in range(neg):
            c = pool[np.random.randint(len(pool))]
            if c == s["target"]:
                continue
            X.append(pair_features(s, c)); y.append(0)
    return np.array(X, dtype=float), np.array(y)

Xc_tr, yc_tr = build_classic_xy(train_samples)
print("고전 ML 학습셋:", Xc_tr.shape, "| 양성 비율:", round(yc_tr.mean(), 3))
print("피처 예시:", Xc_tr[0])

In [ ]:
# 노트북 5 (고전 ML) - 셀 2: RF·로지스틱·KNN 학습 + 동일 403 평가
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# 로지스틱·KNN은 스케일 민감 → 표준화 (RF는 불필요하나 같은 입력 써도 무방)
scaler = StandardScaler().fit(Xc_tr)
Xc_tr_s = scaler.transform(Xc_tr)

print("학습 중...")
rf = RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=0)
rf.fit(Xc_tr, yc_tr)   # RF는 원본 피처

logreg = LogisticRegression(max_iter=1000)
logreg.fit(Xc_tr_s, yc_tr)   # 표준화 피처

knn = KNeighborsClassifier(n_neighbors=15, n_jobs=-1)
knn.fit(Xc_tr_s, yc_tr)   # 표준화 피처
print("학습 완료.")

# --- 고전 ML 랭커: 후보 5000개 각각 점수내서 정렬 ---
def make_classic_ranker(model, use_scaler):
    def rank_fn(sample):
        sample["_uc"] = user_visit_counter(sample["history"])
        # 모든 후보(top_pois)에 대해 피처 생성 → 정답확률 점수
        feats = np.array([pair_features(sample, c) for c in top_pois], dtype=float)
        if use_scaler:
            feats = scaler.transform(feats)
        proba = model.predict_proba(feats)[:, 1]  # P(정답=1)
        order = np.argsort(-proba)
        return [top_pois[i] for i in order]
    return rank_fn

print("\n=== 동일 403 집합: 고전 ML vs NN 최종 비교 ===")
print("인기순     :", evaluate(rank_popular, eval_samples))
print("개인 이력  :", evaluate(rank_personal, eval_samples))
print("RF         :", evaluate(make_classic_ranker(rf, False), eval_samples))
print("로지스틱   :", evaluate(make_classic_ranker(logreg, True), eval_samples))
print("KNN        :", evaluate(make_classic_ranker(knn, True), eval_samples))
print("NN L1(MLP) :", evaluate(rank_nn, eval_samples))
print("NN L2(+이력):", evaluate(rank_nn2, eval_samples))